# A Quick Introduction to Optuna

This Jupyter notebook goes through the basic usage of Optuna.

- Install Optuna
- Write a training algorithm that involves hyperparameters
  - Read train/valid data
  - Define and train model
  - Evaluate model
- Use Optuna to tune the hyperparameters (hyperparameter optimization, HPO)
- Visualize HPO

## Install `optuna`

Optuna can be installed via `pip` or `conda`.

In [1]:
!pip install --quiet optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 10.3 MB/s eta 0:00:00


In [2]:
import optuna

optuna.__version__

'4.3.0'

## Optimize Hyperparameters

### Define a simple scikit-learn model

We start with a simple random forest model to classify flowers in the Iris dataset. We define a function called `objective` that encapsulates the whole training process and outputs the accuracy of the model.

In [3]:
import sklearn.datasets
import sklearn.ensemble
import sklearn.model_selection


def objective():
    iris = sklearn.datasets.load_iris()  # Prepare the data.

    clf = sklearn.ensemble.RandomForestClassifier(n_estimators=5, max_depth=3)  # Define the model.

    return sklearn.model_selection.cross_val_score(
        clf, iris.data, iris.target, n_jobs=-1, cv=3
    ).mean()  # Train and evaluate the model.


print("Accuracy: {}".format(objective()))

Accuracy: 0.9466666666666667


### Optimize hyperparameters of the model

The hyperparameters of the above algorithm are `n_estimators` and `max_depth` for which we can try different values to see if the model accuracy can be improved. The `objective` function is modified to accept a trial object. This trial has several methods for sampling hyperparameters. We create a study to run the hyperparameter optimization and finally read the best hyperparameters.

In [4]:
import optuna


def objective(trial):
    iris = sklearn.datasets.load_iris()

    n_estimators = trial.suggest_int("n_estimators", 2, 20)
    max_depth = int(trial.suggest_float("max_depth", 1, 32, log=True))

    clf = sklearn.ensemble.RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth)

    return sklearn.model_selection.cross_val_score(
        clf, iris.data, iris.target, n_jobs=-1, cv=3
    ).mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

trial = study.best_trial

print("Accuracy: {}".format(trial.value))
print("Best hyperparameters: {}".format(trial.params))

[I 2025-05-15 17:05:00,795] A new study created in memory with name: no-name-bb2a4c29-a24c-49b0-8545-fbf344c912a4
[I 2025-05-15 17:05:00,877] Trial 0 finished with value: 0.9533333333333333 and parameters: {'n_estimators': 14, 'max_depth': 6.720785818188851}. Best is trial 0 with value: 0.9533333333333333.
[I 2025-05-15 17:05:00,937] Trial 1 finished with value: 0.9666666666666667 and parameters: {'n_estimators': 9, 'max_depth': 17.616080773177522}. Best is trial 1 with value: 0.9666666666666667.
[I 2025-05-15 17:05:00,997] Trial 2 finished with value: 0.9533333333333333 and parameters: {'n_estimators': 10, 'max_depth': 13.509082761315069}. Best is trial 1 with value: 0.9666666666666667.
[I 2025-05-15 17:05:01,067] Trial 3 finished with value: 0.9533333333333333 and parameters: {'n_estimators': 10, 'max_depth': 12.597736485656995}. Best is trial 1 with value: 0.9666666666666667.
[I 2025-05-15 17:05:01,097] Trial 4 finished with value: 0.94 and parameters: {'n_estimators': 2, 'max_depth

Accuracy: 0.9733333333333333
Best hyperparameters: {'n_estimators': 7, 'max_depth': 6.60479669607867}


It is possible to condition hyperparameters using Python `if` statements. We can for instance include another classifier, a support vector machine, in our HPO and define hyperparameters specific to the random forest model and the support vector machine.

In [5]:
import sklearn.svm


def objective(trial):
    iris = sklearn.datasets.load_iris()

    classifier = trial.suggest_categorical("classifier", ["RandomForest", "SVC"])

    if classifier == "RandomForest":
        n_estimators = trial.suggest_int("n_estimators", 2, 20)
        max_depth = int(trial.suggest_float("max_depth", 1, 32, log=True))

        clf = sklearn.ensemble.RandomForestClassifier(
            n_estimators=n_estimators, max_depth=max_depth
        )
    else:
        c = trial.suggest_float("svc_c", 1e-10, 1e10, log=True)

        clf = sklearn.svm.SVC(C=c, gamma="auto")

    return sklearn.model_selection.cross_val_score(
        clf, iris.data, iris.target, n_jobs=-1, cv=3
    ).mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

trial = study.best_trial

print("Accuracy: {}".format(trial.value))
print("Best hyperparameters: {}".format(trial.params))

[I 2025-05-15 17:05:34,961] A new study created in memory with name: no-name-030dc4fe-4f92-460e-bead-7ec85588ec9b
[I 2025-05-15 17:05:35,003] Trial 0 finished with value: 0.96 and parameters: {'classifier': 'RandomForest', 'n_estimators': 3, 'max_depth': 3.3449991406792776}. Best is trial 0 with value: 0.96.
[I 2025-05-15 17:05:35,041] Trial 1 finished with value: 0.9666666666666667 and parameters: {'classifier': 'RandomForest', 'n_estimators': 3, 'max_depth': 19.942111041023022}. Best is trial 1 with value: 0.9666666666666667.
[I 2025-05-15 17:05:35,101] Trial 2 finished with value: 0.9533333333333333 and parameters: {'classifier': 'RandomForest', 'n_estimators': 9, 'max_depth': 4.706240398796868}. Best is trial 1 with value: 0.9666666666666667.
[I 2025-05-15 17:05:35,212] Trial 3 finished with value: 0.9666666666666667 and parameters: {'classifier': 'RandomForest', 'n_estimators': 20, 'max_depth': 14.523155382965134}. Best is trial 1 with value: 0.9666666666666667.
[I 2025-05-15 17:0

Accuracy: 0.9733333333333333
Best hyperparameters: {'classifier': 'RandomForest', 'n_estimators': 19, 'max_depth': 10.086521491247984}


### Plotting the study

Plotting the optimization history of the study.

In [6]:
optuna.visualization.plot_optimization_history(study)

Plotting the accuracies for each hyperparameter for each trial.

In [7]:
optuna.visualization.plot_slice(study)

Plotting the accuracy surface for the hyperparameters involved in the random forest model.

In [8]:
optuna.visualization.plot_contour(study, params=["n_estimators", "max_depth"])